# Khám phá dữ liệu: NYC Taxi Lakehouse (Trino)

Notebook này kết nối tới **Trino** để truy vấn trực tiếp vào **Nessie Catalog** (các bảng Iceberg). Việc này giúp phân tích các chỉ số KPIs một cách nhanh chóng với SQL chuẩn.

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# Kết nối tới Trino
engine = create_engine('trino://admin@trino:8080/nessie')
print("Đã kết nối thành công tới Trino!")

✅ Đã kết nối thành công tới Trino!


## 1. Phân tích lượt đi và doanh thu theo Khu Vực (Top 10)

In [2]:
query_zone = '''
    SELECT 
        l.zone_name,
        SUM(f.total_trips) AS total_trips,
        SUM(f.total_revenue) AS total_revenue
    FROM nessie.gold.fact_revenue_by_zone f
    JOIN nessie.gold.dim_location l
      ON f.pulocation_id = l.location_id
    GROUP BY l.zone_name
    ORDER BY total_trips DESC
    LIMIT 10
'''
df_zone = pd.read_sql(query_zone, engine)
display(df_zone)

,zone_name,total_trips,total_revenue
0,JFK Airport,513899,4.106748e+07
1,LaGuardia Airport,434316,2.888410e+07
2,Midtown Center,382946,1.348607e+07
3,Times Sq/Theatre District,350594,1.375775e+07
4,East Village,333301,9.510806e+06
5,Upper East Side South,305850,8.548140e+06
6,East Chelsea,295186,1.029294e+07
7,Upper East Side North,294300,8.154535e+06
8,Union Sq,291959,9.020625e+06
9,Midtown South,270145,9.387159e+06


## 2. Xu hướng chuyến đi theo thời gian (Daily Trips Trend)

In [3]:
query_daily = '''
    SELECT 
        CAST(trip_date AS DATE) AS trip_date,
        trip_type,
        SUM(total_trips) AS total_trips
    FROM nessie.gold.fact_daily_trips
    GROUP BY CAST(trip_date AS DATE), trip_type
    ORDER BY trip_date ASC
'''
df_daily = pd.read_sql(query_daily, engine)

# Biểu diễn dạng bảng pivot để dễ nhìn xu hướng các loại xe
df_pivot = df_daily.pivot(index='trip_date', columns='trip_type', values='total_trips').fillna(0)
display(df_pivot)

trip_type,fhv,green,hvfhv,yellow
trip_date,,,,
2024-01-01,2847,1113,638224,74827
2024-01-02,8178,1544,478162,72336
2024-01-03,9454,1678,498232,79184
2024-01-04,9539,1972,572534,99059
2024-01-05,9141,2000,656526,99014
2024-01-06,4818,1508,708913,92873
2024-01-07,3647,1062,542684,64582
2024-01-08,9823,1821,519477,76894
2024-01-09,10109,1933,624219,89437


## 3. Thói Quen Thanh Toán (Payment Methods)

In [4]:
query_payment = '''
    SELECT 
        p.description AS payment_type,
        SUM(f.total_trips) AS total_trips
    FROM nessie.gold.fact_payment_summary f
    LEFT JOIN nessie.gold.dim_payment_type p
      ON f.payment_type_id = p.payment_type_id
    GROUP BY p.description
    ORDER BY total_trips DESC
'''
df_payment = pd.read_sql(query_payment, engine)
display(df_payment)

,payment_type,total_trips
0,Unknown,19926173
1,Credit Card,2307810
2,Cash,433119
3,Flex Fare trip,115219
4,Dispute,22781
5,No Charge,10088


## 4. Báo Cáo Tổng Hợp Tháng (Monthly Summary KPI)

In [5]:
query_monthly = '''
    SELECT 
        year, 
        month,
        SUM(total_trips) AS total_trips,
        SUM(total_revenue) AS total_revenue
    FROM nessie.gold.fact_monthly_summary
    GROUP BY year, month
    ORDER BY year DESC, month DESC
'''
df_monthly = pd.read_sql(query_monthly, engine)
display(df_monthly)

,year,month,total_trips,total_revenue
0,2024,1,22815190,6.561398e+08
